# 02: Policy-Change Robustness Experiment

**Core Research Hypothesis:** When organizational policies change, retrieval-grounded governance (System B) adapts dynamically by simply re-indexing the updated policy documents without modifying code, whereas hand-encoded rule-based governance (System A) silently goes stale and fails unless engineers manually update and redeploy code.

This notebook demonstrates:
1. Policy document edits across HR, Finance, and IT Access.
2. Static Code Failure rate of System A ($55.6\%$ stale errors).
3. Zero-Code Adaptation of System B.
4. Statistical evaluation using McNemar's paired test and Bootstrap CIs.

In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

results_path = PROJECT_ROOT / "results" / "policy_edit_results.json"
with open(results_path, "r", encoding="utf-8") as f:
    edit_data = json.load(f)

print("Loaded policy edit results.")

## 1. Post-Edit Accuracy & Stale Code Degradation

In [ ]:
sys_a_post = edit_data["overall_system_a"]
sys_b_post = edit_data["overall_system_b"]
mcnemar = edit_data["mcnemar_test"]

print(f"System A (Static Rule Code) Post-Edit Accuracy: {sys_a_post['mean']*100:.1f}% [95% CI: {sys_a_post['ci_lower']*100:.1f}% - {sys_a_post['ci_upper']*100:.1f}%]")
print(f"System B (Zero-Code Re-indexed) Post-Edit Accuracy: {sys_b_post['mean']*100:.1f}% [95% CI: {sys_b_post['ci_lower']*100:.1f}% - {sys_b_post['ci_upper']*100:.1f}%]")
print(f"System A Stale Error Rate: {(1.0 - sys_a_post['mean'])*100:.1f}%")
print(f"McNemar Test p-value: {mcnemar['p_value']}")

## 2. Visualization: Pre- vs Post-Edit Robustness

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5), dpi=150)
labels = ['Pre-Edit Baseline', 'Post-Policy-Edit (Stale Code vs Re-index)']
x = np.arange(len(labels))
width = 0.35

a_vals = [92.6, sys_a_post['mean'] * 100]
b_vals = [66.7, sys_b_post['mean'] * 100]

rects1 = ax.bar(x - width/2, a_vals, width, label='System A (Rule Code)', color='#2b5c8f')
rects2 = ax.bar(x + width/2, b_vals, width, label='System B (Retrieval-Grounded)', color='#27ae60')

ax.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title('Policy-Change Robustness: Zero-Code Dynamic Adaptation vs Stale Code Failure', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11, fontweight='bold')
ax.set_ylim(0, 115)
ax.legend(loc='upper right')
ax.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()